# Alert Triage & Runbook Retrieval**Objective** — Build an intelligent alert-triage system that:1. Ingests raw monitoring alerts (metrics, logs, health-checks)2. Classifies severity and assigns priority3. Summarises context from recent telemetry and change history4. Retrieves the most relevant runbook steps via semantic search5. Proposes concrete next-step actions with safety and escalation guidanceThis notebook demonstrates the full pipeline with synthetic but realisticproduction alert data, evaluates retrieval quality and triage accuracy,and discusses safety guardrails essential for any automation that toucheslive infrastructure.

## Why Automated Alert Triage MattersModern production systems generate thousands of alerts per day.Without intelligent triage, on-call engineers face:| Problem | Impact ||---|---|| **Alert fatigue** | Critical signals buried in noise || **Slow MTTR** | Manual runbook lookup wastes minutes || **Inconsistent response** | Different engineers take different paths || **Context switching** | Correlating dashboards, logs, changes is manual || **Escalation delays** | Unclear when to page senior on-call |An AI triage agent addresses every row:```Raw Alert ──► Severity Classifier ──► Context Summariser                                           │                                    Runbook Retriever                                           │                                    Action Proposer ──► Safety Gate ──► Output```

In [ ]:
import jsonimport hashlibimport mathimport reimport textwrapimport warningsfrom collections import Counter, defaultdictfrom dataclasses import dataclass, fieldfrom datetime import datetime, timedeltafrom typing import Optionalimport numpy as npimport plotly.express as pximport plotly.graph_objects as gofrom plotly.subplots import make_subplotswarnings.filterwarnings('ignore')print('Libraries loaded.')

## 1 · Alert CorpusWe model alerts as structured events from common monitoring stacks(Prometheus/Alertmanager, Datadog, PagerDuty). Each alert has:- **source** — originating monitor (e.g. `prometheus`, `cloudwatch`)- **title** — short description- **body** — raw payload with metric values, tags, and labels- **labels** — key/value pairs: `service`, `env`, `region`- **fired_at** — timestampWe create 24 realistic alerts across 6 services and 4 severity tiers.

In [ ]:
@dataclassclass Alert:    alert_id: str    source: str    title: str    body: str    labels: dict    fired_at: str    raw_severity: str = 'unknown'   # as reported by the monitor    def text(self) -> str:        label_str = ' '.join(f'{k}={v}' for k, v in self.labels.items())        return f'{self.title} | {self.body} | {label_str}'# ── Generate alert corpus ────────────────────────────────────────────BASE = datetime(2026, 4, 12, 2, 0, 0)ALERT_DEFS = [    # (source, title, body, labels, raw_severity, minute_offset)    ('prometheus', 'HighErrorRate',     'HTTP 5xx rate is 12.4% on api-gateway (threshold 1%)',     {'service': 'api-gateway', 'env': 'prod', 'region': 'us-east-1'},     'critical', 0),    ('prometheus', 'HighErrorRate',     'HTTP 5xx rate is 8.7% on payments-svc (threshold 1%)',     {'service': 'payments-svc', 'env': 'prod', 'region': 'us-east-1'},     'critical', 1),    ('cloudwatch', 'CPUUtilization',     'CPU at 94% on i-0abc123 (payments-svc) for 10 min',     {'service': 'payments-svc', 'env': 'prod', 'region': 'us-east-1'},     'warning', 2),    ('datadog', 'LatencyP99Spike',     'p99 latency 4200ms on checkout-svc (baseline 350ms)',     {'service': 'checkout-svc', 'env': 'prod', 'region': 'us-east-1'},     'critical', 3),    ('prometheus', 'PodCrashLooping',     'Pod checkout-svc-7f8b9-xk2lp restarted 8 times in 15 min',     {'service': 'checkout-svc', 'env': 'prod', 'region': 'us-east-1'},     'critical', 4),    ('pagerduty', 'DiskSpaceLow',     'Disk usage 91% on /data volume (db-primary-01)',     {'service': 'postgres-primary', 'env': 'prod', 'region': 'us-east-1'},     'warning', 5),    ('cloudwatch', 'RDSConnectionCount',     'Active connections 485/500 on prod-postgres',     {'service': 'postgres-primary', 'env': 'prod', 'region': 'us-east-1'},     'warning', 6),    ('prometheus', 'MemoryPressure',     'Node us-east-1a-worker-03 memory at 96%, OOMKill imminent',     {'service': 'k8s-node', 'env': 'prod', 'region': 'us-east-1'},     'critical', 7),    ('datadog', 'QueueDepthHigh',     'SQS queue order-events depth 48,200 (threshold 5,000)',     {'service': 'order-processor', 'env': 'prod', 'region': 'us-east-1'},     'warning', 8),    ('prometheus', 'CertExpiringSoon',     'TLS cert for api.example.com expires in 3 days',     {'service': 'ingress', 'env': 'prod', 'region': 'us-east-1'},     'warning', 10),    ('cloudwatch', 'HealthCheckFailed',     'ALB health check failing for target group payments-tg (3/5 unhealthy)',     {'service': 'payments-svc', 'env': 'prod', 'region': 'us-east-1'},     'critical', 11),    ('datadog', 'AnomalousDeploy',     'Deployment checkout-svc v2.14.3 rolled out 4 min ago; error rate rising',     {'service': 'checkout-svc', 'env': 'prod', 'region': 'us-east-1'},     'info', 5),    ('prometheus', 'HighErrorRate',     'HTTP 5xx rate is 0.3% on user-svc (threshold 1%)',     {'service': 'user-svc', 'env': 'prod', 'region': 'eu-west-1'},     'info', 12),    ('datadog', 'LatencyP99Spike',     'p99 latency 520ms on user-svc (baseline 180ms)',     {'service': 'user-svc', 'env': 'staging', 'region': 'eu-west-1'},     'info', 14),    ('prometheus', 'HighErrorRate',     'HTTP 5xx rate is 15.1% on payments-svc (threshold 1%)',     {'service': 'payments-svc', 'env': 'prod', 'region': 'us-east-1'},     'critical', 20),    ('cloudwatch', 'LambdaThrottling',     'Lambda email-sender throttled 340 invocations in 5 min',     {'service': 'email-sender', 'env': 'prod', 'region': 'us-east-1'},     'warning', 22),    ('prometheus', 'NodeNotReady',     'Node us-east-1b-worker-07 NotReady for 3 min',     {'service': 'k8s-node', 'env': 'prod', 'region': 'us-east-1'},     'critical', 25),    ('datadog', 'SlowQuery',     'Query avg 12.4s on prod-postgres (table: orders, full scan detected)',     {'service': 'postgres-primary', 'env': 'prod', 'region': 'us-east-1'},     'warning', 28),    ('pagerduty', 'SSLHandshakeFailure',     'SSL handshake failures spiking on api-gateway from region ap-south-1',     {'service': 'api-gateway', 'env': 'prod', 'region': 'ap-south-1'},     'warning', 30),    ('prometheus', 'HighErrorRate',     'gRPC error rate 6.2% on inventory-svc (threshold 0.5%)',     {'service': 'inventory-svc', 'env': 'prod', 'region': 'us-east-1'},     'critical', 32),    ('cloudwatch', 'EBSIOPSExhausted',     'EBS volume vol-0def456 IOPS credits exhausted on db-replica-02',     {'service': 'postgres-replica', 'env': 'prod', 'region': 'us-east-1'},     'warning', 35),    ('datadog', 'KafkaConsumerLag',     'Consumer group order-processor lag 120,000 on topic order-events',     {'service': 'order-processor', 'env': 'prod', 'region': 'us-east-1'},     'warning', 38),    ('prometheus', 'ContainerOOMKilled',     'Container checkout-svc OOMKilled, memory limit 512Mi exceeded',     {'service': 'checkout-svc', 'env': 'prod', 'region': 'us-east-1'},     'critical', 40),    ('pagerduty', 'DNSResolutionFailure',     'DNS resolution failing for payments.internal.example.com from us-east-1',     {'service': 'payments-svc', 'env': 'prod', 'region': 'us-east-1'},     'critical', 42),]alerts = []for i, (src, title, body, labels, sev, offset) in enumerate(ALERT_DEFS):    alerts.append(Alert(        alert_id=f'ALT-{i+1:04d}',        source=src,        title=title,        body=body,        labels=labels,        fired_at=(BASE + timedelta(minutes=offset)).isoformat(),        raw_severity=sev,    ))print(f'Alert corpus: {len(alerts)} alerts')print(f'Sources     : {Counter(a.source for a in alerts)}')print(f'Services    : {len(set(a.labels["service"] for a in alerts))} unique')print(f'Severities  : {Counter(a.raw_severity for a in alerts)}')

## 2 · Runbook Knowledge BaseReal SRE teams maintain runbooks — step-by-step remediation guides forknown failure modes. We build a searchable knowledge base of 18 runbookentries covering the services in our alert corpus.Each runbook has:- **runbook_id** — unique identifier- **title** — failure scenario- **tags** — services, symptoms, categories- **steps** — ordered remediation steps- **escalation** — when and whom to escalate to- **safety_notes** — things NOT to do, blast-radius warnings

In [ ]:
@dataclassclass Runbook:    runbook_id: str    title: str    tags: list    steps: list    escalation: str    safety_notes: str    def text(self) -> str:        tag_str = ', '.join(self.tags)        step_str = ' '.join(f'({i+1}) {s}' for i, s in enumerate(self.steps))        return (f'{self.title} | Tags: {tag_str} | Steps: {step_str} '                f'| Escalation: {self.escalation} | Safety: {self.safety_notes}')RUNBOOK_DEFS = [    ('RB-001', 'High HTTP Error Rate (5xx)',     ['api-gateway', 'payments-svc', 'error-rate', '5xx', 'http'],     ['Check recent deployments in the affected service',      'Inspect application logs for stack traces (kubectl logs / CloudWatch)',      'Verify upstream dependencies are healthy (database, cache, queues)',      'If deployment-related: initiate rollback via CI/CD pipeline',      'If dependency issue: check dependency-specific runbook',      'Monitor error rate for 5 min after remediation'],     'Page senior on-call if error rate >10% for >5 min after rollback',     'Do NOT restart pods before capturing logs. Rollback via pipeline, not manual kubectl.'),    ('RB-002', 'Pod CrashLooping',     ['kubernetes', 'crashloop', 'pod', 'restart', 'oom'],     ['Get pod events: kubectl describe pod <name>',      'Check exit code: OOMKilled (137) vs application error (1)',      'If OOMKilled: check memory limits vs actual usage',      'If application error: inspect logs from previous container',      'Check if recent config change or deployment triggered it',      'If OOM: bump memory limit temporarily, file capacity ticket'],     'Escalate to platform team if >3 pods crash-looping across services',     'Do NOT delete the pod before capturing describe output and logs.'),    ('RB-003', 'CPU Saturation on Instance',     ['cpu', 'saturation', 'ec2', 'instance', 'cloudwatch'],     ['Identify top CPU consumers: top -b -n1 or pidstat',      'Check if caused by legitimate load increase or runaway process',      'If runaway: kill the process, investigate root cause',      'If load increase: check if autoscaling is active',      'Consider vertical or horizontal scaling'],     'Escalate if CPU >90% across all instances in the ASG for >10 min',     'Do NOT terminate the instance if it hosts a primary database.'),    ('RB-004', 'High Latency (P99 Spike)',     ['latency', 'p99', 'slow', 'response-time', 'checkout'],     ['Check distributed traces for the slowest endpoints',      'Look for slow database queries in APM',      'Verify connection pool saturation',      'Check for lock contention or deadlocks',      'If deployment-related: rollback and compare latency',      'Check downstream service latency for cascading issues'],     'Page database DBA if slow queries are the root cause and >5 min',     'Avoid restarting services during peak traffic without draining connections first.'),    ('RB-005', 'Disk Space Low',     ['disk', 'storage', 'volume', 'space', 'postgres'],     ['Identify largest directories: du -sh /* | sort -rh | head',      'Check for unrotated logs, WAL segments, or temp files',      'Clean up: rotate logs, vacuum database, purge old backups',      'If database WAL: check replication lag or stuck slots',      'Expand volume if cleanup is insufficient'],     'Escalate to DBA if disk >95% on a database volume',     'Do NOT delete WAL segments manually — use pg_archivecleanup. '     'Do NOT resize root volume without a snapshot.'),    ('RB-006', 'Database Connection Exhaustion',     ['database', 'connections', 'postgres', 'rds', 'pool'],     ['Check current connections: SELECT count(*) FROM pg_stat_activity',      'Identify idle-in-transaction sessions',      'Kill long-running idle sessions if safe',      'Review application connection pool settings (min, max, timeout)',      'Check for connection leaks in recent code changes',      'Consider PgBouncer if not already in place'],     'Escalate to DBA if connections hit max and queries are queuing',     'Do NOT restart the database to free connections — kill idle sessions first.'),    ('RB-007', 'Node Memory Pressure / OOMKill',     ['memory', 'oom', 'oomkill', 'node', 'kubernetes', 'pressure'],     ['Check which pods are consuming the most memory on the node',      'Verify memory requests vs limits for top consumers',      'Cordon the node to prevent new scheduling',      'Drain non-critical pods to other nodes',      'Investigate memory leak if one pod grows continuously',      'File capacity ticket if cluster is under-provisioned'],     'Page platform team if >2 nodes are in MemoryPressure state',     'Do NOT drain database pods or stateful workloads without checking replication status.'),    ('RB-008', 'Queue Depth / Consumer Lag',     ['queue', 'sqs', 'kafka', 'lag', 'consumer', 'depth', 'backpressure'],     ['Check consumer health: are consumers running and processing?',      'Look for consumer errors in logs',      'Check if message format changed (schema mismatch)',      'Scale up consumer instances or concurrency',      'If SQS: check DLQ for poison messages',      'Monitor lag trend: is it growing or stabilizing?'],     'Escalate if consumer lag doubles within 10 min or DLQ grows',     'Do NOT purge the queue — messages will be lost permanently.'),    ('RB-009', 'TLS/SSL Certificate Expiring',     ['tls', 'ssl', 'certificate', 'expiry', 'ingress', 'https'],     ['Verify current cert expiry: openssl s_client -connect host:443',      'Check cert-manager or ACME renewal logs for errors',      'If auto-renewal failed: renew manually via cert-manager or ACM',      'Update the cert in the ingress/load-balancer',      'Verify renewal: curl -vI https://host and check dates'],     'Escalate to security team if cert expires in <24 hours and renewal fails',     'Do NOT deploy a self-signed cert to production as a workaround.'),    ('RB-010', 'ALB/NLB Health Check Failures',     ['alb', 'nlb', 'health-check', 'target-group', 'unhealthy'],     ['Check which targets are unhealthy in the target group',      'Verify the health check endpoint returns 200',      'Check security groups and NACLs for connectivity',      'Inspect application logs on unhealthy targets',      'If all targets unhealthy: check shared dependency (DB, config)'],     'Escalate if >50% targets unhealthy for >3 min',     'Do NOT de-register healthy targets to "reset" the group.'),    ('RB-011', 'Recent Deployment Rollback',     ['deployment', 'rollback', 'release', 'ci-cd', 'canary'],     ['Identify the exact version/commit that was deployed',      'Compare metrics before vs after deployment',      'Initiate rollback through CI/CD pipeline (not kubectl)',      'Verify rollback completed: check running image/version',      'Notify the owning team with deployment details',      'Block the pipeline until root cause is identified'],     'Escalate to engineering lead if rollback fails or symptoms persist',     'Never force-push images. Always use the pipeline. Tag the failed deploy for post-mortem.'),    ('RB-012', 'Lambda Throttling',     ['lambda', 'throttle', 'concurrency', 'serverless', 'invocation'],     ['Check current reserved vs unreserved concurrency',      'Review CloudWatch for invocation count trend',      'Identify triggering source: is it a traffic spike or retry storm?',      'Increase reserved concurrency if within account limits',      'If retry storm: fix the upstream caller, add backoff',      'Consider provisioned concurrency for latency-sensitive functions'],     'Escalate to cloud team if account-level concurrency limit is reached',     'Do NOT set concurrency to 0 (disables the function). Check downstream impact before scaling up.'),    ('RB-013', 'Kubernetes Node NotReady',     ['node', 'notready', 'kubernetes', 'kubelet', 'unreachable'],     ['Check node conditions: kubectl describe node <name>',      'Verify kubelet status: systemctl status kubelet on the node',      'Check for kernel/OS-level issues: dmesg, journalctl',      'If network issue: check VPC/security group/ENI',      'If hardware issue: cordon, drain, and replace the node',      'Verify workloads rescheduled to healthy nodes'],     'Escalate to platform team if >1 node goes NotReady simultaneously',     'Do NOT force-delete pods on NotReady nodes — they may still be running.'),    ('RB-014', 'Slow Database Queries',     ['database', 'slow-query', 'postgres', 'index', 'full-scan', 'explain'],     ['Identify the slow query from pg_stat_statements or APM',      'Run EXPLAIN ANALYZE on the query (on a replica if possible)',      'Check for missing indexes, seq scans on large tables',      'Check for table bloat and run VACUUM ANALYZE if needed',      'If new query: review with the owning team for optimization',      'Add appropriate indexes, test on staging first'],     'Escalate to DBA if query is a critical path and >10s avg for >5 min',     'Do NOT add indexes on production without DBA review — could lock tables.'),    ('RB-015', 'SSL Handshake Failures',     ['ssl', 'handshake', 'tls', 'certificate', 'cipher', 'protocol'],     ['Check cert chain validity from the client perspective',      'Verify TLS version compatibility (TLS 1.2+ required)',      'Check for cipher suite mismatches',      'Inspect CDN/WAF configuration for SSL termination issues',      'If region-specific: check regional cert deployment',      'Test with openssl s_client from multiple regions'],     'Escalate to security team if affecting >1 region or >5% of traffic',     'Do NOT downgrade TLS version to fix the issue. Check cert deployment instead.'),    ('RB-016', 'gRPC Error Rate High',     ['grpc', 'error', 'unavailable', 'deadline-exceeded', 'internal'],     ['Check gRPC error codes: UNAVAILABLE vs DEADLINE_EXCEEDED vs INTERNAL',      'UNAVAILABLE: check service discovery and load balancer',      'DEADLINE_EXCEEDED: check downstream latency and timeout settings',      'INTERNAL: inspect application logs for stack traces',      'Verify service mesh (Envoy/Istio) sidecar health if applicable',      'Check for recent protobuf schema changes'],     'Escalate if error rate >5% for >5 min on a critical-path service',     'Do NOT change timeout values in production without load testing.'),    ('RB-017', 'EBS IOPS Exhaustion',     ['ebs', 'iops', 'storage', 'throughput', 'io', 'volume'],     ['Check CloudWatch BurstBalance and VolumeQueueLength',      'Identify the workload causing high I/O (iotop, pidstat -d)',      'If burst credits exhausted: volume type needs upgrading (gp2->gp3/io2)',      'Check for vacuum/reindex/backup operations consuming I/O',      'Migrate to provisioned IOPS volume type if recurring'],     'Escalate to DBA if IOPS exhaustion is on a production database volume',     'Do NOT detach EBS volumes from running instances. Modify volume type online.'),    ('RB-018', 'DNS Resolution Failure',     ['dns', 'resolution', 'route53', 'internal', 'service-discovery'],     ['Test resolution from affected hosts: dig/nslookup the failing hostname',      'Check Route53 / CoreDNS health and logs',      'Verify VPC DHCP options and DNS settings',      'Check if the service endpoint was recently changed',      'Flush DNS cache on affected nodes if stale entries suspected',      'Verify security groups allow DNS traffic (UDP/TCP 53)'],     'Page network team immediately — DNS failures cascade across all services',     'Do NOT restart CoreDNS pods without checking if it is a capacity issue — could worsen the outage.'),]runbooks = []for rb_id, title, tags, steps, esc, safety in RUNBOOK_DEFS:    runbooks.append(Runbook(        runbook_id=rb_id, title=title, tags=tags,        steps=steps, escalation=esc, safety_notes=safety,    ))print(f'Runbook KB: {len(runbooks)} runbooks')for rb in runbooks[:5]:    print(f'  {rb.runbook_id}: {rb.title} ({len(rb.steps)} steps)')

## 3 · Lightweight Embedding EngineFor semantic retrieval, we build a simple TF-IDF–inspired embedding.In production you would use a dense encoder (e.g. `sentence-transformers`or an API embedding model). Our bag-of-words approach still demonstratesthe retrieval pipeline clearly.**How it works:**1. Tokenise → lowercase → remove punctuation → split on whitespace2. Build vocabulary from the runbook corpus3. Compute TF-IDF vectors for each runbook and each alert4. Rank by cosine similarity

In [ ]:
class TFIDFEmbedder:    """Minimal TF-IDF embedder for demonstration."""    def __init__(self):        self.vocab: dict[str, int] = {}        self.idf: np.ndarray = np.array([])        self._fitted = False    @staticmethod    def _tokenise(text: str) -> list[str]:        text = re.sub(r'[^a-z0-9\s]', ' ', text.lower())        return [t for t in text.split() if len(t) > 1]    def fit(self, docs: list[str]):        df = Counter()        for doc in docs:            unique_tokens = set(self._tokenise(doc))            for t in unique_tokens:                df[t] += 1        self.vocab = {t: i for i, t in enumerate(sorted(df.keys()))}        n = len(docs)        self.idf = np.array([            math.log((n + 1) / (df[t] + 1)) + 1 for t in sorted(df.keys())        ])        self._fitted = True        return self    def transform(self, text: str) -> np.ndarray:        vec = np.zeros(len(self.vocab))        tokens = self._tokenise(text)        tf = Counter(tokens)        for t, count in tf.items():            if t in self.vocab:                vec[self.vocab[t]] = count        vec = vec * self.idf        norm = np.linalg.norm(vec)        if norm > 0:            vec /= norm        return vecdef cosine_sim(a: np.ndarray, b: np.ndarray) -> float:    dot = np.dot(a, b)    na, nb = np.linalg.norm(a), np.linalg.norm(b)    return float(dot / (na * nb)) if na > 0 and nb > 0 else 0.0# Fit on runbook corpusembedder = TFIDFEmbedder()rb_texts = [rb.text() for rb in runbooks]embedder.fit(rb_texts)rb_vectors = [embedder.transform(t) for t in rb_texts]print(f'Vocabulary size : {len(embedder.vocab)}')print(f'Runbook vectors : {len(rb_vectors)} x {len(rb_vectors[0])}')

## 4 · Severity ClassificationThe raw severity from monitoring tools is a starting point, but triagerequires contextual re-classification. Our classifier combines:| Signal | Weight | Rationale ||---|---|---|| Raw severity from monitor | 0.3 | Baseline intent of the alert rule || Environment (prod vs staging) | 0.25 | Prod issues are always higher priority || Service criticality tier | 0.25 | Payment services > internal tools || Keyword severity signals | 0.2 | "OOMKill", "failure", "exhausted" escalate |**Output tiers:** `P1-Critical`, `P2-High`, `P3-Medium`, `P4-Low`### Safety Principle: Conservative Classification> When uncertain, classify **up** (higher severity). A false-alarm P1> costs an interruption. A missed P1 costs an outage.

In [ ]:
SERVICE_TIERS = {    'api-gateway': 1, 'payments-svc': 1, 'checkout-svc': 1,    'postgres-primary': 1,    'order-processor': 2, 'inventory-svc': 2, 'k8s-node': 2,    'user-svc': 3, 'email-sender': 3, 'ingress': 2,    'postgres-replica': 2,}SEVERITY_SCORES = {'critical': 1.0, 'warning': 0.6, 'info': 0.2, 'unknown': 0.4}ESCALATION_KEYWORDS = [    'oomkill', 'failure', 'exhausted', 'failing', 'crashed',    'unreachable', 'notready', 'expired', 'throttled', 'lost',]@dataclassclass TriageResult:    alert: Alert    priority: str          # P1..P4    score: float           # 0..1    signals: dict          # breakdown of scoring signals    context_summary: str = ''    matched_runbooks: list = field(default_factory=list)    proposed_actions: list = field(default_factory=list)    escalation_note: str = ''    safety_warnings: list = field(default_factory=list)def classify_severity(alert: Alert) -> tuple[str, float, dict]:    """Multi-signal severity classifier."""    signals = {}    # Signal 1: raw severity    raw_score = SEVERITY_SCORES.get(alert.raw_severity, 0.4)    signals['raw_severity'] = raw_score    # Signal 2: environment    env = alert.labels.get('env', 'unknown')    env_score = {'prod': 1.0, 'staging': 0.4, 'dev': 0.1}.get(env, 0.3)    signals['environment'] = env_score    # Signal 3: service criticality    svc = alert.labels.get('service', '')    tier = SERVICE_TIERS.get(svc, 3)    tier_score = {1: 1.0, 2: 0.7, 3: 0.4}.get(tier, 0.4)    signals['service_tier'] = tier_score    # Signal 4: keyword escalation    text_lower = alert.body.lower()    kw_hits = [kw for kw in ESCALATION_KEYWORDS if kw in text_lower]    kw_score = min(1.0, len(kw_hits) * 0.4) if kw_hits else 0.0    signals['keyword_escalation'] = kw_score    signals['matched_keywords'] = kw_hits    # Weighted combination    final = (0.30 * raw_score + 0.25 * env_score             + 0.25 * tier_score + 0.20 * kw_score)    # Conservative: round up at boundaries    if final >= 0.75:        priority = 'P1-Critical'    elif final >= 0.55:        priority = 'P2-High'    elif final >= 0.35:        priority = 'P3-Medium'    else:        priority = 'P4-Low'    return priority, round(final, 3), signals# Classify all alertstriage_results = []for alert in alerts:    pri, score, sigs = classify_severity(alert)    triage_results.append(TriageResult(        alert=alert, priority=pri, score=score, signals=sigs    ))priority_counts = Counter(tr.priority for tr in triage_results)print('Severity classification complete:')for p in ['P1-Critical', 'P2-High', 'P3-Medium', 'P4-Low']:    print(f'  {p}: {priority_counts.get(p, 0)}')print(f'\nSample P1 alert:')p1 = next(tr for tr in triage_results if tr.priority == 'P1-Critical')print(f'  {p1.alert.alert_id}: {p1.alert.title} — {p1.alert.body}')print(f'  Score: {p1.score} | Signals: {p1.signals}')

## 5 · Context SummarisationBefore proposing actions, we enrich each alert with surrounding context.In production, this pulls from:- **Deployment history** — recent deploys to the affected service- **Correlated alerts** — other alerts on the same service within a window- **Change log** — recent infra/config changesWe simulate this by correlating our alert corpus: grouping alerts byservice and looking for temporal clusters.

In [ ]:
def build_context_summary(tr: TriageResult, all_results: list[TriageResult],                          window_min: int = 15) -> str:    """Build a context summary by correlating nearby alerts."""    alert = tr.alert    fired = datetime.fromisoformat(alert.fired_at)    svc = alert.labels.get('service', '')    # Find correlated alerts (same service, within time window)    correlated = []    for other in all_results:        if other.alert.alert_id == alert.alert_id:            continue        other_svc = other.alert.labels.get('service', '')        other_time = datetime.fromisoformat(other.alert.fired_at)        if other_svc == svc and abs((other_time - fired).total_seconds()) < window_min * 60:            correlated.append(other)    # Find related alerts (different service, same region, within window)    region = alert.labels.get('region', '')    related = []    for other in all_results:        if other.alert.alert_id == alert.alert_id:            continue        o_svc = other.alert.labels.get('service', '')        o_region = other.alert.labels.get('region', '')        o_time = datetime.fromisoformat(other.alert.fired_at)        if (o_svc != svc and o_region == region                and abs((o_time - fired).total_seconds()) < window_min * 60):            related.append(other)    lines = [f'Alert {alert.alert_id} on {svc} ({alert.labels.get("env", "?")}/',             f'{alert.labels.get("region", "?")}):']    lines.append(f'  Symptom: {alert.body}')    if correlated:        lines.append(f'  Correlated ({len(correlated)} alerts on same service):')        for c in correlated[:3]:            lines.append(f'    - {c.alert.title}: {c.alert.body}')    if related:        lines.append(f'  Related ({len(related)} alerts in same region):')        for r in related[:3]:            r_svc = r.alert.labels.get('service', '?')            lines.append(f'    - [{r_svc}] {r.alert.title}: {r.alert.body}')    if not correlated and not related:        lines.append('  No correlated or related alerts in the time window.')    return '\n'.join(lines)# Build context for all triage resultsfor tr in triage_results:    tr.context_summary = build_context_summary(tr, triage_results)# Show a samplesample = next(tr for tr in triage_results              if tr.priority == 'P1-Critical'              and tr.alert.labels.get('service') == 'payments-svc')print(sample.context_summary)

## 6 · Runbook RetrievalFor each alert, we retrieve the top-K most relevant runbooks usingcosine similarity between the alert text embedding and the pre-computedrunbook embeddings.We also apply a **tag-boost**: if the alert's service or keywords matcha runbook's tags, we add a bonus to the similarity score. This hybridapproach (semantic + structured) is standard in production retrieval.### Retrieval Quality Matters for Safety> Retrieving the *wrong* runbook is worse than retrieving none.> A wrong runbook can lead an engineer to take harmful actions> (e.g. applying a database runbook to a networking issue).> We set a **minimum similarity threshold** to filter low-confidence> matches and flag them for human review.

In [ ]:
TAG_BOOST = 0.15MIN_SIMILARITY = 0.08TOP_K = 3def retrieve_runbooks(alert: Alert, top_k: int = TOP_K                      ) -> list[tuple[Runbook, float]]:    """Retrieve top-K runbooks for an alert, with tag boost."""    alert_vec = embedder.transform(alert.text())    scored = []    alert_tokens = set(TFIDFEmbedder._tokenise(alert.text()))    svc = alert.labels.get('service', '').lower()    for rb, rb_vec in zip(runbooks, rb_vectors):        sim = cosine_sim(alert_vec, rb_vec)        # Tag boost: direct service match or keyword overlap        tag_set = set(t.lower() for t in rb.tags)        boost = 0.0        if svc in tag_set or svc.split('-')[0] in tag_set:            boost += TAG_BOOST        keyword_overlap = len(alert_tokens & tag_set)        if keyword_overlap > 0:            boost += min(TAG_BOOST, keyword_overlap * 0.05)        final_score = sim + boost        if final_score >= MIN_SIMILARITY:            scored.append((rb, round(final_score, 4)))    scored.sort(key=lambda x: x[1], reverse=True)    return scored[:top_k]# Retrieve for all alertsfor tr in triage_results:    tr.matched_runbooks = retrieve_runbooks(tr.alert)# Show retrieval for first few P1 alertsprint('Runbook retrieval results (P1 alerts):\n')for tr in triage_results:    if tr.priority != 'P1-Critical':        continue    print(f'{tr.alert.alert_id}: {tr.alert.title} — {tr.alert.body[:60]}...')    for rb, score in tr.matched_runbooks:        print(f'  ► {rb.runbook_id}: {rb.title} (score={score})')    print()

## 7 · Safety Framework & Escalation PolicyAutomated triage systems operate near live production infrastructure.Safety is not optional — it is the primary design constraint.### Three Safety Layers```┌─────────────────────────────────────────────────────────────┐│  Layer 1: PREVENT — Stop harmful actions before they happen │├─────────────────────────────────────────────────────────────┤│  Layer 2: DETECT — Flag uncertainty and low-confidence      │├─────────────────────────────────────────────────────────────┤│  Layer 3: CONTAIN — Limit blast radius when things go wrong │└─────────────────────────────────────────────────────────────┘```| Layer | Mechanism | Example ||---|---|---|| **Prevent** | Deny-list of destructive actions | Never auto-execute `DROP`, `DELETE`, `kubectl delete` || **Prevent** | Require human approval for P1 | P1 actions displayed but not auto-executed || **Detect** | Confidence thresholds | Flag runbook matches below 0.15 similarity || **Detect** | Anomaly detection on triage output | Alert if >50% of alerts are classified P1 || **Contain** | Scope limits | Only suggest actions for the specific service, not cluster-wide || **Contain** | Dry-run mode | Propose commands with `--dry-run` flag first |### Escalation Matrix| Priority | Response Time | Who | Auto-Actions Allowed ||---|---|---|---|| P1-Critical | Immediate | Senior on-call + engineering lead | Display only — human executes || P2-High | 5 min | Primary on-call | Safe diagnostics only (read-only commands) || P3-Medium | 30 min | On-call during business hours | Diagnostics + safe remediations || P4-Low | Next business day | Ticket to owning team | Full auto-remediation allowed |### Key Principle: Suggest, Don't Execute (for critical systems)> The triage agent's role is to **accelerate** human decision-making,> not to **replace** it. For P1/P2 alerts, the agent surfaces context> and runbook steps so the on-call engineer can act faster —> but the engineer always makes the final call.

## 8 · Action ProposerThe action proposer synthesises triage signals, context, and retrievedrunbooks into a structured response with:1. **Situation summary** — what's happening and how severe2. **Proposed steps** — ordered from the best-matching runbook3. **Escalation guidance** — when to page and whom4. **Safety warnings** — things to avoid5. **Confidence indicator** — how sure the system is about the match

In [ ]:
CONFIDENCE_THRESHOLDS = {    'high': 0.25,    'medium': 0.15,    'low': 0.0,}DESTRUCTIVE_PATTERNS = [    'delete', 'drop', 'purge', 'truncate', 'force-delete',    'rm -rf', 'terminate', 'destroy',]def propose_actions(tr: TriageResult) -> TriageResult:    """Generate proposed actions from triage result."""    actions = []    safety_warnings = []    escalation_note = ''    if not tr.matched_runbooks:        actions.append('⚠ No matching runbook found. Manual investigation required.')        safety_warnings.append('LOW CONFIDENCE: No automated guidance available. '                               'Escalate to senior on-call.')        escalation_note = 'Immediate escalation — no runbook match.'        tr.proposed_actions = actions        tr.safety_warnings = safety_warnings        tr.escalation_note = escalation_note        return tr    best_rb, best_score = tr.matched_runbooks[0]    # Determine confidence level    if best_score >= CONFIDENCE_THRESHOLDS['high']:        confidence = 'HIGH'    elif best_score >= CONFIDENCE_THRESHOLDS['medium']:        confidence = 'MEDIUM'    else:        confidence = 'LOW'    # Add confidence header    actions.append(f'Confidence: {confidence} (score={best_score}, '                   f'runbook={best_rb.runbook_id})')    # Add runbook steps, with safety annotations    for i, step in enumerate(best_rb.steps):        step_lower = step.lower()        is_destructive = any(p in step_lower for p in DESTRUCTIVE_PATTERNS)        if is_destructive:            actions.append(f'  Step {i+1} [⚠ REQUIRES APPROVAL]: {step}')            safety_warnings.append(f'Step {i+1} is potentially destructive — '                                   f'requires human approval before execution.')        elif tr.priority in ('P1-Critical', 'P2-High'):            actions.append(f'  Step {i+1} [SUGGEST]: {step}')        else:            actions.append(f'  Step {i+1}: {step}')    # Add escalation guidance from runbook    escalation_note = best_rb.escalation    if confidence == 'LOW':        escalation_note += ' | LOW CONFIDENCE MATCH — verify runbook applicability.'    # Add safety notes from runbook    safety_warnings.append(f'Runbook safety: {best_rb.safety_notes}')    # P1 always gets an extra safety note    if tr.priority == 'P1-Critical':        safety_warnings.append('P1 POLICY: All remediation steps require human '                               'confirmation. Do not auto-execute.')    tr.proposed_actions = actions    tr.safety_warnings = safety_warnings    tr.escalation_note = escalation_note    return tr# Generate actions for all triage resultsfor tr in triage_results:    propose_actions(tr)# Show complete triage output for a P1 alertsample = next(tr for tr in triage_results if tr.priority == 'P1-Critical')print(f'═══ TRIAGE REPORT: {sample.alert.alert_id} ═══')print(f'Title   : {sample.alert.title}')print(f'Priority: {sample.priority} (score={sample.score})')print(f'\n── Context ──')print(sample.context_summary)print(f'\n── Proposed Actions ──')for action in sample.proposed_actions:    print(action)print(f'\n── Escalation ──')print(sample.escalation_note)print(f'\n── Safety Warnings ──')for sw in sample.safety_warnings:    print(f'  ⚠ {sw}')

## 9 · Full Pipeline: Batch Triage ReportWe now run the complete pipeline on all 24 alerts and generate astructured triage report that an on-call team would receive.Alerts are sorted by priority (P1 first), and grouped by serviceto highlight correlated failures.

In [ ]:
def format_triage_report(results: list[TriageResult]) -> str:    """Format a complete triage report."""    sorted_results = sorted(results, key=lambda r: r.score, reverse=True)    lines = ['╔══════════════════════════════════════════════╗',             '║       ALERT TRIAGE REPORT                   ║',             f'║  Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}              ║',             f'║  Alerts: {len(results)}  |  '             f'P1: {sum(1 for r in results if r.priority=="P1-Critical")}  |  '             f'P2: {sum(1 for r in results if r.priority=="P2-High")}  |  '             f'P3: {sum(1 for r in results if r.priority=="P3-Medium")}  |  '             f'P4: {sum(1 for r in results if r.priority=="P4-Low")}      ║',             '╚══════════════════════════════════════════════╝', '']    # Group by service    svc_groups = defaultdict(list)    for r in sorted_results:        svc_groups[r.alert.labels.get('service', 'unknown')].append(r)    # Sort service groups by highest priority alert    svc_order = sorted(svc_groups.keys(),                       key=lambda s: max(r.score for r in svc_groups[s]),                       reverse=True)    for svc in svc_order:        group = svc_groups[svc]        max_pri = min(r.priority for r in group)  # P1 < P2 lexicographically        lines.append(f'┌── {svc.upper()} ({len(group)} alerts, highest: {max_pri}) ──')        for r in group:            lines.append(f'│  {r.alert.alert_id} [{r.priority}] {r.alert.title}')            lines.append(f'│    {r.alert.body[:80]}')            if r.matched_runbooks:                rb, score = r.matched_runbooks[0]                lines.append(f'│    → Runbook: {rb.runbook_id} — {rb.title} '                             f'(confidence={score})')            else:                lines.append(f'│    → ⚠ No matching runbook')        lines.append('└' + '─' * 50)        lines.append('')    return '\n'.join(lines)report = format_triage_report(triage_results)print(report)

## 10 · EvaluationWe evaluate the triage system on three axes:1. **Classification accuracy** — does the priority match expert judgment?2. **Retrieval precision** — does the top runbook match the expected one?3. **Safety coverage** — are all P1 alerts flagged with safety warnings?

In [ ]:
# Ground truth mapping: alert_id -> expected priority, expected runbookGROUND_TRUTH = {    'ALT-0001': ('P1-Critical', 'RB-001'),  # High 5xx on api-gateway    'ALT-0002': ('P1-Critical', 'RB-001'),  # High 5xx on payments    'ALT-0003': ('P2-High',     'RB-003'),  # CPU on payments    'ALT-0004': ('P1-Critical', 'RB-004'),  # Latency on checkout    'ALT-0005': ('P1-Critical', 'RB-002'),  # CrashLoop checkout    'ALT-0006': ('P2-High',     'RB-005'),  # Disk low    'ALT-0007': ('P2-High',     'RB-006'),  # DB connections    'ALT-0008': ('P1-Critical', 'RB-007'),  # Memory pressure    'ALT-0009': ('P2-High',     'RB-008'),  # Queue depth    'ALT-0010': ('P2-High',     'RB-009'),  # Cert expiring    'ALT-0011': ('P1-Critical', 'RB-010'),  # ALB health check    'ALT-0012': ('P2-High',     'RB-011'),  # Deploy anomaly    'ALT-0013': ('P4-Low',      'RB-001'),  # Low 5xx user-svc    'ALT-0014': ('P4-Low',      'RB-004'),  # Latency staging    'ALT-0015': ('P1-Critical', 'RB-001'),  # High 5xx payments (2nd)    'ALT-0016': ('P2-High',     'RB-012'),  # Lambda throttle    'ALT-0017': ('P1-Critical', 'RB-013'),  # Node NotReady    'ALT-0018': ('P2-High',     'RB-014'),  # Slow query    'ALT-0019': ('P2-High',     'RB-015'),  # SSL handshake    'ALT-0020': ('P1-Critical', 'RB-016'),  # gRPC errors    'ALT-0021': ('P2-High',     'RB-017'),  # EBS IOPS    'ALT-0022': ('P2-High',     'RB-008'),  # Kafka lag    'ALT-0023': ('P1-Critical', 'RB-002'),  # OOMKilled    'ALT-0024': ('P1-Critical', 'RB-018'),  # DNS failure}# Evaluate classificationclf_correct = 0clf_total = 0clf_details = []for tr in triage_results:    if tr.alert.alert_id in GROUND_TRUTH:        expected_pri, expected_rb = GROUND_TRUTH[tr.alert.alert_id]        clf_total += 1        match = tr.priority == expected_pri        if match:            clf_correct += 1        else:            clf_details.append({                'alert': tr.alert.alert_id,                'expected': expected_pri,                'predicted': tr.priority,                'score': tr.score,            })# Evaluate retrievalret_correct = 0ret_total = 0ret_in_top3 = 0for tr in triage_results:    if tr.alert.alert_id in GROUND_TRUTH:        _, expected_rb = GROUND_TRUTH[tr.alert.alert_id]        ret_total += 1        if tr.matched_runbooks:            top_rb = tr.matched_runbooks[0][0].runbook_id            if top_rb == expected_rb:                ret_correct += 1            # Check top-3            top3_ids = [rb.runbook_id for rb, _ in tr.matched_runbooks[:3]]            if expected_rb in top3_ids:                ret_in_top3 += 1# Safety coveragep1_results = [tr for tr in triage_results if tr.priority == 'P1-Critical']p1_with_safety = sum(1 for tr in p1_results if tr.safety_warnings)print('═══ EVALUATION RESULTS ═══\n')print(f'Classification Accuracy: {clf_correct}/{clf_total} '      f'({clf_correct/clf_total*100:.1f}%)')if clf_details:    print(f'  Misclassifications:')    for d in clf_details:        print(f'    {d["alert"]}: expected {d["expected"]}, '              f'got {d["predicted"]} (score={d["score"]})')print(f'\nRetrieval Precision@1: {ret_correct}/{ret_total} '      f'({ret_correct/ret_total*100:.1f}%)')print(f'Retrieval Recall@3   : {ret_in_top3}/{ret_total} '      f'({ret_in_top3/ret_total*100:.1f}%)')print(f'\nSafety Coverage:')print(f'  P1 with safety warnings: {p1_with_safety}/{len(p1_results)} '      f'({p1_with_safety/len(p1_results)*100:.0f}%)')

## 11 · Visualisations

In [ ]:
# Viz 1: Priority distributionpri_data = []for tr in triage_results:    pri_data.append({        'Priority': tr.priority,        'Service': tr.alert.labels.get('service', 'unknown'),        'Score': tr.score,    })import pandas as pddf_pri = pd.DataFrame(pri_data)fig1 = px.histogram(    df_pri, x='Priority', color='Priority',    color_discrete_map={        'P1-Critical': '#e74c3c', 'P2-High': '#f39c12',        'P3-Medium': '#3498db', 'P4-Low': '#2ecc71'    },    category_orders={'Priority': ['P1-Critical', 'P2-High', 'P3-Medium', 'P4-Low']},    title='Alert Priority Distribution',)fig1.update_layout(showlegend=False, height=400)fig1.show()

In [ ]:
# Viz 2: Service × Priority heatmapsvc_pri = defaultdict(lambda: defaultdict(int))for tr in triage_results:    svc = tr.alert.labels.get('service', 'unknown')    svc_pri[svc][tr.priority] += 1services = sorted(svc_pri.keys())priorities = ['P1-Critical', 'P2-High', 'P3-Medium', 'P4-Low']heat_matrix = [[svc_pri[svc][p] for p in priorities] for svc in services]fig2 = go.Figure(data=go.Heatmap(    z=heat_matrix, x=priorities, y=services,    colorscale='YlOrRd', text=heat_matrix, texttemplate='%{text}',    hovertemplate='Service: %{y}<br>Priority: %{x}<br>Count: %{z}<extra></extra>',))fig2.update_layout(    title='Service × Priority Heatmap',    height=450, xaxis_title='Priority', yaxis_title='Service',)fig2.show()

In [ ]:
# Viz 3: Triage score distribution by priorityfig3 = px.strip(    df_pri, x='Priority', y='Score', color='Priority',    color_discrete_map={        'P1-Critical': '#e74c3c', 'P2-High': '#f39c12',        'P3-Medium': '#3498db', 'P4-Low': '#2ecc71'    },    category_orders={'Priority': ['P1-Critical', 'P2-High', 'P3-Medium', 'P4-Low']},    title='Triage Score Distribution by Priority',    hover_data=['Service'],)# Add threshold linesfor y_val, label in [(0.75, 'P1 threshold'), (0.55, 'P2 threshold'),                      (0.35, 'P3 threshold')]:    fig3.add_hline(y=y_val, line_dash='dash', line_color='gray',                   annotation_text=label, annotation_position='right')fig3.update_layout(height=400, showlegend=False)fig3.show()

In [ ]:
# Viz 4: Alert × Runbook similarity heatmapsim_matrix = []alert_labels_list = []for tr in triage_results:    alert_vec = embedder.transform(tr.alert.text())    row = []    for rb_vec in rb_vectors:        row.append(round(cosine_sim(alert_vec, rb_vec), 3))    sim_matrix.append(row)    alert_labels_list.append(f'{tr.alert.alert_id}: {tr.alert.title[:20]}')rb_labels = [f'{rb.runbook_id}: {rb.title[:25]}' for rb in runbooks]fig4 = go.Figure(data=go.Heatmap(    z=sim_matrix, x=rb_labels, y=alert_labels_list,    colorscale='Viridis',    hovertemplate='Alert: %{y}<br>Runbook: %{x}<br>Similarity: %{z:.3f}<extra></extra>',))fig4.update_layout(    title='Alert × Runbook Cosine Similarity',    height=700, xaxis_tickangle=45,    margin=dict(b=150, r=50),)fig4.show()

In [ ]:
# Viz 5: Signal breakdown radar for P1 alertsp1_alerts = [tr for tr in triage_results if tr.priority == 'P1-Critical']categories = ['Raw Severity', 'Environment', 'Service Tier', 'Keyword Escalation']signal_keys = ['raw_severity', 'environment', 'service_tier', 'keyword_escalation']fig5 = go.Figure()for tr in p1_alerts[:5]:  # Limit to 5 for readability    values = [tr.signals.get(k, 0) for k in signal_keys]    values.append(values[0])  # Close the polygon    fig5.add_trace(go.Scatterpolar(        r=values,        theta=categories + [categories[0]],        name=f'{tr.alert.alert_id}: {tr.alert.title[:20]}',        fill='toself', opacity=0.5,    ))fig5.update_layout(    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),    title='P1 Alert Signal Breakdown (Radar)',    height=500,)fig5.show()

In [ ]:
# Viz 6: Alert timeline with priority colour codingtimeline_data = []for tr in triage_results:    timeline_data.append({        'Time': tr.alert.fired_at,        'Service': tr.alert.labels.get('service', 'unknown'),        'Alert': tr.alert.title,        'Priority': tr.priority,        'Score': tr.score,        'Body': tr.alert.body[:60],    })df_tl = pd.DataFrame(timeline_data)df_tl['Time'] = pd.to_datetime(df_tl['Time'])fig6 = px.scatter(    df_tl, x='Time', y='Service', color='Priority',    size='Score', size_max=18,    color_discrete_map={        'P1-Critical': '#e74c3c', 'P2-High': '#f39c12',        'P3-Medium': '#3498db', 'P4-Low': '#2ecc71'    },    hover_data=['Alert', 'Body', 'Score'],    title='Alert Timeline by Service',)fig6.update_layout(height=450)fig6.show()

In [ ]:
# Viz 7: Evaluation metrics summarymetrics = {    'Classification\nAccuracy': clf_correct / clf_total * 100,    'Retrieval\nPrecision@1': ret_correct / ret_total * 100,    'Retrieval\nRecall@3': ret_in_top3 / ret_total * 100,    'Safety\nCoverage': p1_with_safety / len(p1_results) * 100,}fig7 = go.Figure(data=[    go.Bar(        x=list(metrics.keys()),        y=list(metrics.values()),        text=[f'{v:.1f}%' for v in metrics.values()],        textposition='outside',        marker_color=['#3498db', '#e74c3c', '#f39c12', '#2ecc71'],    )])fig7.update_layout(    title='Triage System Evaluation Metrics',    yaxis_title='Score (%)', yaxis_range=[0, 110],    height=400, showlegend=False,)fig7.show()

## 12 · Production Architecture### Complete System Design```                    ┌──────────────┐                    │ Alertmanager │                    │  / PagerDuty │                    │  / Datadog   │                    └──────┬───────┘                           │ webhook                    ┌──────▼───────┐                    │ Alert Ingest │                    │   (Kafka)    │                    └──────┬───────┘                           │          ┌────────────────┼────────────────┐          │                │                │   ┌──────▼──────┐ ┌──────▼──────┐ ┌───────▼──────┐   │  Severity   │ │  Context    │ │   Runbook    │   │ Classifier  │ │ Enricher    │ │  Retriever   │   │ (ML model)  │ │ (telemetry) │ │ (vector DB)  │   └──────┬──────┘ └──────┬──────┘ └───────┬──────┘          │                │                │          └────────────────┼────────────────┘                           │                    ┌──────▼───────┐                    │   Action     │                    │  Proposer    │                    │   (LLM)     │                    └──────┬───────┘                           │                    ┌──────▼───────┐                    │ Safety Gate  │                    │ (deny-list,  │                    │  approvals)  │                    └──────┬───────┘                           │              ┌────────────┼────────────┐              │            │            │       ┌──────▼───┐ ┌─────▼────┐ ┌─────▼────┐       │  Slack   │ │ PagerDuty│ │ Auto-    │       │ Channel  │ │ Incident │ │ Remediate│       │ (P3/P4)  │ │ (P1/P2)  │ │ (P4 only)│       └──────────┘ └──────────┘ └──────────┘```### Key Production Components| Component | Technology | Purpose ||---|---|---|| Alert ingestion | Kafka / SQS | Decouple alerting from processing || Severity classifier | Fine-tuned BERT or rule engine | Multi-signal classification || Context enricher | Datadog/Grafana APIs | Fetch recent metrics, logs, deploys || Runbook retriever | Vector DB (Pinecone, Weaviate) | Semantic search over runbook corpus || Action proposer | LLM (GPT-4 / Claude) | Generate human-readable action plans || Safety gate | Rule engine + approval workflow | Block destructive / high-risk actions || Output channels | Slack, PagerDuty, JIRA | Route by priority tier |

### Production Code: LangChain ImplementationBelow is a reference implementation using LangChain with a vectorstore for runbook retrieval and an LLM for action proposal generation.

In [ ]:
PRODUCTION_CODE = r'''# ── production_triage.py ──────────────────────────────────────────────from langchain_openai import ChatOpenAI, OpenAIEmbeddingsfrom langchain_community.vectorstores import Chromafrom langchain.prompts import ChatPromptTemplatefrom langchain.schema import Documentfrom pydantic import BaseModel, Fieldclass TriageResponse(BaseModel):    """Structured output from the triage agent."""    priority: str = Field(description="P1-Critical, P2-High, P3-Medium, or P4-Low")    situation_summary: str = Field(description="1-2 sentence situation summary")    proposed_steps: list[str] = Field(description="Ordered remediation steps")    escalation: str = Field(description="Escalation guidance")    safety_warnings: list[str] = Field(description="Safety warnings and caveats")    confidence: float = Field(ge=0.0, le=1.0, description="Confidence score")class AlertTriageAgent:    def __init__(self, runbook_dir: str = "./runbooks"):        self.llm = ChatOpenAI(model="gpt-4o", temperature=0)        self.embeddings = OpenAIEmbeddings(model="text-embedding-3-small")        # Load runbooks into vector store        self.vectorstore = Chroma(            collection_name="runbooks",            embedding_function=self.embeddings,            persist_directory="./chroma_runbooks",        )        self.triage_prompt = ChatPromptTemplate.from_messages([            ("system", """You are an SRE triage agent. Given an alert and            relevant runbook context, produce a structured triage response.            SAFETY RULES:            1. Never suggest destructive actions (delete, drop, purge)               without explicit [REQUIRES APPROVAL] tag            2. For P1-Critical: only SUGGEST actions, never auto-execute            3. If runbook match confidence is below 0.3, flag for human review            4. Always include service-specific safety notes from the runbook"""),            ("human", """Alert: {alert_text}            Context: {context}            Runbook matches: {runbook_context}            Classify priority and propose actions."""),        ])    def triage(self, alert: dict) -> TriageResponse:        # Retrieve relevant runbooks        alert_text = f"{alert['title']}: {alert['body']}"        docs = self.vectorstore.similarity_search_with_score(            alert_text, k=3        )        runbook_context = "\n".join(            f"[score={1-score:.2f}] {doc.page_content}"            for doc, score in docs        )        # Generate triage response        chain = self.triage_prompt | self.llm.with_structured_output(            TriageResponse        )        result = chain.invoke({            "alert_text": alert_text,            "context": alert.get("context", "No additional context"),            "runbook_context": runbook_context,        })        return result'''print(PRODUCTION_CODE)

## Summary### What We Built| Component | Description | Key Metric ||---|---|---|| Alert corpus | 24 realistic production alerts across 10 services | 4 severity tiers || Runbook KB | 18 SRE runbooks with steps, escalation, and safety notes | 18 failure modes || Severity classifier | Multi-signal (raw severity + env + tier + keywords) | Classification accuracy || Context summariser | Temporal correlation and service grouping | 15-min correlation window || Runbook retriever | TF-IDF with tag boost, minimum similarity threshold | Precision@1 and Recall@3 || Action proposer | Structured output with confidence and safety annotations | Safety coverage |### Safety & Escalation — Key Takeaways1. **Conservative classification** — when uncertain, classify up (P3→P2)2. **Suggest, don't execute** — for P1/P2, humans always make the final call3. **Deny-list destructive actions** — never auto-execute DELETE, DROP, PURGE4. **Confidence thresholds** — flag low-confidence runbook matches for review5. **Runbook safety notes** — every runbook includes "what NOT to do"6. **Escalation matrix** — clear ownership and response times per priority tier7. **Blast-radius containment** — scope actions to the affected service only### Production Path- Replace TF-IDF with dense embeddings (`sentence-transformers` or API)- Add LLM-based action proposer for natural language output- Integrate with real monitoring APIs (Datadog, Prometheus, PagerDuty)- Store runbooks in a version-controlled knowledge base with CI/CD updates- Add feedback loop: on-call engineers rate triage quality → fine-tune classifier